# Experimento: Região de Compatibilidade de Marginais Quânticas para 3 Qubits

Este notebook investiga numericamente a diferença entre:

1. **Consistência na sobreposição A**: $\|\text{Tr}_B(\rho_{AB}) - \text{Tr}_C(\rho_{AC})\|_F < \varepsilon$
2. **Existência de extensão global**: factibilidade do SDP para $\rho_{ABC}$

**Objetivo**: Verificar numericamente que a consistência na sobreposição é condição *necessária*, mas **não suficiente**, para a existência de uma extensão global $\rho_{ABC}$.

---

## Estrutura
- **Célula 1**: Imports e reutilização das funções do notebook `inicial_teste.ipynb`
- **Célula 2**: Função auxiliar — geração de matrizes densidade aleatórias
- **Célula 3**: Funções auxiliares — construção de marginais correlacionadas com marginal A fixa
- **Célula 4**: Experimento A — marginais completamente aleatórias
- **Célula 5**: Experimento B — marginais construídas para satisfazer a condição de sobreposição
- **Célula 6**: Caso especial — exemplo de Bell (validação)
- **Célula 7**: Estatísticas consolidadas
- **Célula 8**: Visualizações


In [ ]:
# ==============================================================================
# Célula 1 — Imports e reutilização das funções do notebook inicial_teste.ipynb
# ==============================================================================
# NOTA: Este notebook depende que o notebook inicial_teste.ipynb tenha sido
# executado na mesma sessão de kernel, ou que as funções abaixo sejam
# redefinidas aqui. Para garantir portabilidade, todas as funções necessárias
# são importadas/redefinidas a seguir.

from __future__ import annotations

import time
import sys
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Bibliotecas SpaceCore e SDPLab para Programação Semidefinida
import spacecore
from spacecore import Context, DenseVectorSpace, HermitianSpace, NumpyOps
import sdplab
from sdplab import DenseConstraintOp, SDPProblem
from sdplab.solvers import run_cvxpy_solver

np.set_printoptions(precision=4, suppress=True)

print(f"Versão SDPLab:    {sdplab.__version__}")
print(f"Versão SpaceCore: {spacecore.__version__}")

# ──────────────────────────────────────────────────────────────────────────────
# Re-definição das funções do projeto (idênticas ao inicial_teste.ipynb)
# Não foram modificadas — apenas copiadas para autocontenção deste notebook.
# ──────────────────────────────────────────────────────────────────────────────

# Qubits da base computacional
zero = np.array([1.0, 0.0], dtype=complex)
one  = np.array([0.0, 1.0], dtype=complex)


def ketbra(psi: np.ndarray) -> np.ndarray:
    """
    Constrói o operador projetor |psi><psi| para um vetor de estado |psi>.
    (Reutilizado de inicial_teste.ipynb, sem modificações.)
    """
    psi = np.asarray(psi, dtype=complex).reshape(-1)
    return np.outer(psi, psi.conj())


def tensor(*args: np.ndarray) -> np.ndarray:
    """
    Calcula o produto tensorial sequencial (Kronecker) de múltiplos
    vetores ou matrizes.
    (Reutilizado de inicial_teste.ipynb, sem modificações.)
    """
    result = args[0]
    for a in args[1:]:
        result = np.kron(result, a)
    return result


def partial_trace(
    rho: np.ndarray,
    keep: Optional[Sequence[int]] = None,
    trace_out: Optional[Sequence[int]] = None,
    dims: Optional[Sequence[int]] = None,
) -> np.ndarray:
    """
    Traço parcial de uma matriz densidade multiqubit.
    (Reutilizado de inicial_teste.ipynb, sem modificações.)
    """
    rho = np.asarray(rho, dtype=complex)
    if dims is None:
        num_qubits = int(round(np.log2(rho.shape[0])))
        dims = [2] * num_qubits
    dims = tuple(dims)
    N = len(dims)

    if keep is not None and trace_out is not None:
        raise ValueError("Especifique apenas `keep` ou `trace_out`, não ambos.")
    if keep is not None:
        keep_systems = tuple(sorted(keep))
    elif trace_out is not None:
        trace_set = set(trace_out)
        keep_systems = tuple(sorted(i for i in range(N) if i not in trace_set))
    else:
        raise ValueError("É necessário especificar `keep` ou `trace_out`.")

    complement = tuple(i for i in range(N) if i not in keep_systems)

    rho_tensor = rho.reshape(dims + dims)

    bra_indices = list(range(N))
    ket_indices = list(range(N, 2 * N))

    for c in complement:
        ket_indices[c] = bra_indices[c]

    out_bra = [bra_indices[s] for s in keep_systems]
    out_ket = [ket_indices[s] for s in keep_systems]

    reduced_tensor = np.einsum(rho_tensor, bra_indices + ket_indices, out_bra + out_ket)
    d_out = int(np.prod([dims[s] for s in keep_systems]))
    return reduced_tensor.reshape(d_out, d_out)


def base_hermitiana(d: int) -> List[np.ndarray]:
    """
    Base ortonormal para o espaço das matrizes Hermitianas Herm(d).
    (Reutilizado de inicial_teste.ipynb, sem modificações.)
    """
    bases: List[np.ndarray] = []
    for i in range(d):
        E = np.zeros((d, d), dtype=complex)
        E[i, i] = 1.0
        bases.append(E)
    for i in range(d):
        for j in range(i + 1, d):
            E_re = np.zeros((d, d), dtype=complex)
            E_re[i, j] = 1.0 / np.sqrt(2)
            E_re[j, i] = 1.0 / np.sqrt(2)
            bases.append(E_re)
            E_im = np.zeros((d, d), dtype=complex)
            E_im[i, j] = -1j / np.sqrt(2)
            E_im[j, i] = 1j / np.sqrt(2)
            bases.append(E_im)
    return bases


def embed_operator(
    O_s: np.ndarray,
    sistema: Sequence[int],
    dim: Optional[Sequence[int]] = None,
    *,
    dims: Optional[Sequence[int]] = None,
) -> np.ndarray:
    r"""
    Incorpora um operador local O_s atuando no subsistema S (`sistema`) no espaço
    global H = prod_k H_k, inserindo identidades nos subsistemas do complemento S^c.
    (Reutilizado de inicial_teste.ipynb, sem modificações.)
    """
    if dim is None:
        dim = dims
    if dim is None:
        raise ValueError("É necessário especificar as dimensões locais `dim` (ou `dims`).")
    dims_tuple = tuple(dim)
    Nt = len(dims_tuple)
    sistema_tuple = tuple(sistema)
    Ns = len(sistema_tuple)
    complemento = tuple(i for i in range(Nt) if i not in sistema_tuple)

    dimensao_s = tuple(dims_tuple[i] for i in sistema_tuple)
    O_tensor = O_s.reshape(dimensao_s + dimensao_s)

    # Produto externo com identidade para cada sistema complementar
    res = O_tensor
    for c in complemento:
        I_c = np.eye(dims_tuple[c], dtype=O_s.dtype)
        res = np.multiply.outer(res, I_c)

    # Mapeamento dos eixos originais para a ordem global
    axis_map_bra = {s: idx for idx, s in enumerate(sistema_tuple)}
    axis_map_ket = {s: Ns + idx for idx, s in enumerate(sistema_tuple)}
    for idx, c in enumerate(complemento):
        axis_map_bra[c] = 2 * Ns + 2 * idx
        axis_map_ket[c] = 2 * Ns + 2 * idx + 1

    perm = [axis_map_bra[i] for i in range(Nt)] + [axis_map_ket[i] for i in range(Nt)]
    res = np.transpose(res, perm)
    d_total = int(np.prod(dims_tuple))
    return res.reshape(d_total, d_total)


def parse_subsystem_key(key: Union[str, Sequence[int]], num_qubits: int = 3) -> Tuple[int, ...]:
    """
    Normaliza identificadores de subsistemas como 'AB', 'AC', 'BC' ou (0, 1), (0, 2).
    (Reutilizado de inicial_teste.ipynb, sem modificações.)
    """
    if isinstance(key, (tuple, list)):
        return tuple(int(x) for x in key)
    if isinstance(key, str):
        char_map = {chr(ord('A') + i): i for i in range(26)}
        key_upper = key.upper().strip()
        return tuple(char_map[ch] for ch in key_upper if ch in char_map)
    raise TypeError(f"Formato de subsistema inválido: {key!r}")


def build_qmp_sdp(
    marginals_dict: Dict[Union[str, Tuple[int, ...]], np.ndarray],
    dims: Optional[Sequence[int]] = None,
    include_trace_norm: bool = True,
    ctx: Optional[Context] = None,
) -> Tuple[SDPProblem, np.ndarray, np.ndarray]:
    r"""
    Constrói a formulação SDP para o problema de representabilidade de marginais
    quânticas no formato nativo da biblioteca SDPLab.
    (Reutilizado de inicial_teste.ipynb, sem modificações.)
    """
    normalized_marginals: Dict[Tuple[int, ...], np.ndarray] = {}
    for k, v in marginals_dict.items():
        sub_tuple = parse_subsystem_key(k)
        mat = np.asarray(v, dtype=complex)
        if mat.ndim != 2 or mat.shape[0] != mat.shape[1]:
            raise ValueError(f"A marginal {k} deve ser uma matriz quadrada; formato {mat.shape}.")
        normalized_marginals[sub_tuple] = mat

    if dims is None:
        max_idx = max(max(sub) for sub in normalized_marginals.keys())
        dims = [2] * (max_idx + 1)
    dims_tuple = tuple(dims)
    d_total = int(np.prod(dims_tuple))

    M_list: List[np.ndarray] = []
    b_list: List[float] = []

    if include_trace_norm:
        M_list.append(np.eye(d_total, dtype=complex))
        b_list.append(1.0)

    for sistema, omega_S in normalized_marginals.items():
        d_S = omega_S.shape[0]
        expected_d_S = int(np.prod([dims_tuple[s] for s in sistema]))
        if d_S != expected_d_S:
            raise ValueError(
                f"Dimensão da marginal para o subsistema {sistema} é {d_S}, "
                f"mas esperava-se {expected_d_S} conforme dims={dims_tuple}."
            )
        base_S = base_hermitiana(d_S)
        for E in base_S:
            val = float(np.real(np.trace(E @ omega_S)))
            M_global = embed_operator(E, sistema, dims_tuple)
            M_list.append(M_global)
            b_list.append(val)

    M_full = np.stack(M_list, axis=0)
    b_full = np.array(b_list, dtype=float)

    if ctx is None:
        ctx = Context(NumpyOps(), dtype="complex128", check_level="none")

    dom = HermitianSpace(d_total, ctx=ctx)
    cod = DenseVectorSpace((len(b_full),), ctx=ctx)

    A = DenseConstraintOp(np.swapaxes(M_full, -1, -2), dom, cod, ctx)
    sdp = SDPProblem(dom.zeros(), A, b_full, ctx=ctx)

    return sdp, M_full, b_full


def solve_qmp(
    sdp: SDPProblem,
    solver: str = "CLARABEL",
    verbose: bool = False,
    **kwargs: Any,
) -> Dict[str, Any]:
    """
    Resolve o problema SDP de representabilidade de marginais quânticas.
    (Reutilizado de inicial_teste.ipynb, sem modificações.)
    """
    try:
        X, y, prob = run_cvxpy_solver(
            sdp,
            solver=solver,
            verbose=verbose,
            return_problem=True,
            **kwargs,
        )
        status = prob.status
        is_feasible = status in ("optimal", "optimal_inaccurate")
        rho_mat = np.asarray(X)
        rho_mat = (rho_mat + rho_mat.conj().T) / 2.0

        return {
            "status": status,
            "is_feasible": is_feasible,
            "rho": rho_mat,
            "dual_y": np.asarray(y),
            "problem": prob,
            "error_message": None,
        }
    except Exception as err:
        err_str = str(err)
        # Trata SolverError, ValueError e qualquer outra exceção do CVXPY/CLARABEL
        status = "infeasible" if "infeasible" in err_str.lower() else "failed"
        return {
            "status": status,
            "is_feasible": False,
            "rho": None,
            "dual_y": None,
            "problem": None,
            "error_message": err_str,
        }


print("✓ Todas as funções do projeto carregadas com sucesso.")


## Célula 2 — Geração de Matrizes Densidade Aleatórias

Usamos o método de Ginibre: $G$ é uma matriz complexa aleatória,
e $\rho = G G^\dagger / \text{Tr}(G G^\dagger)$.

Isso produz estados distribuídos segundo a medida de Hilbert-Schmidt,
garantindo por construção: $\rho = \rho^\dagger$, $\rho \geq 0$, $\text{Tr}(\rho) = 1$.

In [ ]:
# ==============================================================================
# Célula 2 — Geração de matrizes densidade aleatórias (método de Ginibre)
# ==============================================================================

def random_density_matrix(d: int, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """
    Gera uma matriz densidade aleatória de dimensão d usando o método de Ginibre.

    A construção é:
        G = matriz complexa d×d de entradas i.i.d. N(0,1) + i N(0,1)
        rho = G G† / Tr(G G†)

    Garantias numéricas verificadas:
        - rho = rho†   (hermiticidade)
        - rho >= 0     (positividade semidefinida)
        - Tr(rho) = 1  (normalização)

    Parameters
    ----------
    d : int
        Dimensão do espaço de Hilbert (d=2 para qubit, d=4 para 2 qubits).
    rng : np.random.Generator, opcional
        Gerador de números aleatórios (para reprodutibilidade).

    Returns
    -------
    rho : np.ndarray, shape (d, d), dtype complex128
        Matriz densidade válida.
    """
    if rng is None:
        rng = np.random.default_rng()

    # Gera matriz complexa aleatória
    G = rng.standard_normal((d, d)) + 1j * rng.standard_normal((d, d))

    # Constrói rho = G G†
    rho = G @ G.conj().T

    # Normaliza pelo traço
    rho = rho / np.trace(rho)

    # Garante hermiticidade perfeita numericamente (elimina erros de arredondamento)
    rho = (rho + rho.conj().T) / 2.0

    return rho


def _verify_density_matrix(rho: np.ndarray, tol: float = 1e-10) -> Dict[str, Any]:
    """
    Verifica as propriedades de uma matriz densidade.

    Returns
    -------
    dict com campos: hermitian_error, trace_error, min_eigenvalue, is_valid
    """
    herm_err = float(np.linalg.norm(rho - rho.conj().T, 'fro'))
    trace_err = float(abs(np.trace(rho) - 1.0))
    evals = np.linalg.eigvalsh((rho + rho.conj().T) / 2.0)
    min_eval = float(np.min(evals))
    is_valid = (herm_err < tol) and (trace_err < tol) and (min_eval >= -tol)
    return {
        "hermitian_error": herm_err,
        "trace_error": trace_err,
        "min_eigenvalue": min_eval,
        "is_valid": is_valid,
    }


# ── Teste rápido de sanidade ──────────────────────────────────────────────────
print("Teste de sanidade: random_density_matrix")
rng_test = np.random.default_rng(seed=0)

for d_test in [2, 4]:
    rho_test = random_density_matrix(d_test, rng=rng_test)
    v = _verify_density_matrix(rho_test)
    status_str = "✓ VÁLIDA" if v["is_valid"] else "✗ INVÁLIDA"
    print(
        f"  d={d_test}: {status_str} | "
        f"herm_err={v['hermitian_error']:.2e} | "
        f"trace_err={v['trace_error']:.2e} | "
        f"lambda_min={v['min_eigenvalue']:.4f}"
    )


## Célula 3 — Construção de Marginais com Marginal A Fixa

Para o Experimento B, precisamos construir $\rho_{AB}$ e $\rho_{AC}$ tal que:
$$\text{Tr}_B(\rho_{AB}) = \rho_A \quad \text{e} \quad \text{Tr}_C(\rho_{AC}) = \rho_A$$

**Construção produto (trivial):**
$$\rho_{AB}^{\text{prod}} = \rho_A \otimes \sigma_B, \quad \rho_{AC}^{\text{prod}} = \rho_A \otimes \sigma_C$$

**Construção correlacionada (não-trivial):**  
Usamos a decomposição espectral de $\rho_A$ para construir estados misturados correlacionados via purificações aleatórias.

In [ ]:
# ==============================================================================
# Célula 3 — Funções para construção de marginais com marginal A fixa
# ==============================================================================


def build_product_marginals(
    rho_A: np.ndarray,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Constrói rho_AB = rho_A ⊗ sigma_B e rho_AC = rho_A ⊗ sigma_C,
    onde sigma_B e sigma_C são matrizes densidade aleatórias de dimensão 2.

    Garante: Tr_B(rho_AB) = rho_A  e  Tr_C(rho_AC) = rho_A.
    Note: admite sempre extensão global trivial rho_ABC = rho_A ⊗ sigma_B ⊗ sigma_C.

    Parameters
    ----------
    rho_A : ndarray (2, 2)
        Marginal prescrita para o subsistema A.
    rng : np.random.Generator, opcional

    Returns
    -------
    rho_AB : ndarray (4, 4)
    rho_AC : ndarray (4, 4)
    """
    if rng is None:
        rng = np.random.default_rng()

    sigma_B = random_density_matrix(2, rng=rng)
    sigma_C = random_density_matrix(2, rng=rng)

    rho_AB = np.kron(rho_A, sigma_B)
    rho_AC = np.kron(rho_A, sigma_C)

    return rho_AB, rho_AC


def build_correlated_marginals(
    rho_A: np.ndarray,
    rng: Optional[np.random.Generator] = None,
    num_kraus: int = 3,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Constrói rho_AB e rho_AC correlacionados que mantêm rho_A como marginal.

    Estratégia (purificação aleatória via decomposição espectral):
    1. Decompõe rho_A = sum_k lambda_k |k><k|
    2. Constrói purificação em AB: |Psi_AB> = sum_k sqrt(lambda_k) |k>_A ⊗ |phi_k>_B
       onde |phi_k> são vetores aleatórios ortonormalizados.
    3. O estado puro |Psi_AB><Psi_AB| tem Tr_B(.) = rho_A por construção.
    4. Aplica canal quântico aleatório para obter estado misto correlacionado.

    Parameters
    ----------
    rho_A : ndarray (2, 2)
        Marginal prescrita para o subsistema A.
    rng : np.random.Generator, opcional
    num_kraus : int
        Número de operadores de Kraus para o canal aleatório (controla
        o grau de mistura introduzido).

    Returns
    -------
    rho_AB : ndarray (4, 4)  com Tr_B(rho_AB) ≈ rho_A
    rho_AC : ndarray (4, 4)  com Tr_C(rho_AC) ≈ rho_A
    """
    if rng is None:
        rng = np.random.default_rng()

    def _purification_state(rho: np.ndarray) -> np.ndarray:
        """
        Constrói uma purificação de rho em espaço ampliado d×d.
        Retorna a matriz densidade do estado puro (d^2 × d^2 não, d × d do estado puro
        no espaço AB de dimensão d×d = 4×4 para qubit).
        """
        d = rho.shape[0]
        # Decomposição espectral
        evals, evecs = np.linalg.eigh(rho)
        # Garante não-negatividade (erros numéricos)
        evals = np.maximum(evals, 0.0)

        # Vetores ortonormais aleatórios para o ambiente
        # (aqui o ambiente tem a mesma dimensão que o sistema)
        U_env, _ = np.linalg.qr(
            rng.standard_normal((d, d)) + 1j * rng.standard_normal((d, d))
        )

        # |Psi> = sum_k sqrt(lambda_k) |k>_sys ⊗ U_env|k>_env
        psi = np.zeros(d * d, dtype=complex)
        for k in range(d):
            sys_vec = evecs[:, k]           # autovetor do sistema
            env_vec = U_env[:, k]           # vetor ortonormal do ambiente
            psi += np.sqrt(evals[k]) * np.kron(sys_vec, env_vec)

        # Normaliza (deve já ser 1, mas garante numericamente)
        norm = np.linalg.norm(psi)
        if norm > 1e-14:
            psi = psi / norm

        return ketbra(psi)

    def _apply_random_channel_on_B(rho_AB_pure: np.ndarray) -> np.ndarray:
        """
        Aplica um canal quântico aleatório no subsistema B de rho_AB.
        O canal é descrito por operadores de Kraus aleatórios normalizados.
        Preserva Tr_B(rho_AB) = rho_A.
        """
        d_A, d_B = 2, 2
        d_AB = d_A * d_B

        # Gera operadores de Kraus aleatórios para o canal em B
        # K_i: d_B × d_B tal que sum_i K_i† K_i = I_B
        # Método: gera uma isometria d_B × (d_B * num_kraus) e extrai blocos
        d_out = d_B * num_kraus
        V, _ = np.linalg.qr(
            rng.standard_normal((d_out, d_out)) + 1j * rng.standard_normal((d_out, d_out))
        )
        # Toma as primeiras d_B colunas como isometria d_out × d_B
        V_iso = V[:d_B * num_kraus, :d_B]  # shape (num_kraus * d_B, d_B)

        kraus_ops = [V_iso[i * d_B:(i + 1) * d_B, :] for i in range(num_kraus)]

        # Verifica completeza: sum K_i† K_i = I
        completeness = sum(K.conj().T @ K for K in kraus_ops)
        # Re-normaliza se necessário
        if not np.allclose(completeness, np.eye(d_B), atol=1e-10):
            # Usa SVD para obter operadores de Kraus válidos
            U_q, _, _ = np.linalg.svd(completeness)
            kraus_ops = [K @ U_q.conj().T for K in kraus_ops]
            # Normaliza
            completeness2 = sum(K.conj().T @ K for K in kraus_ops)
            scale = np.sqrt(np.linalg.norm(completeness2, 2))
            kraus_ops = [K / scale for K in kraus_ops]

        # Aplica o canal: rho_AB -> sum_k (I_A ⊗ K_k) rho_AB (I_A ⊗ K_k)†
        rho_out = np.zeros((d_AB, d_AB), dtype=complex)
        for K in kraus_ops:
            # I_A ⊗ K_k
            op = np.kron(np.eye(d_A, dtype=complex), K)
            rho_out += op @ rho_AB_pure @ op.conj().T

        # Renormaliza (preserva Tr)
        tr_out = np.trace(rho_out)
        if abs(tr_out) > 1e-14:
            rho_out = rho_out / tr_out

        # Garante hermiticidade
        rho_out = (rho_out + rho_out.conj().T) / 2.0

        return rho_out

    # Constrói purificação de rho_A em espaço AB
    rho_AB_pure = _purification_state(rho_A)
    # Aplica canal aleatório em B para introduzir correlações diferentes
    rho_AB = _apply_random_channel_on_B(rho_AB_pure)

    # Repete de forma independente para AC
    rho_AC_pure = _purification_state(rho_A)
    rho_AC = _apply_random_channel_on_B(rho_AC_pure)  # canal aleatório independente em C

    return rho_AB, rho_AC


def check_overlap_consistency(
    rho_AB: np.ndarray,
    rho_AC: np.ndarray,
    tol: float = 1e-6,
) -> Tuple[float, bool]:
    """
    Calcula o erro de sobreposição ||Tr_B(rho_AB) - Tr_C(rho_AC)||_F
    e determina se a condição de sobreposição é satisfeita.

    Reutiliza partial_trace do projeto com a assinatura correta:
        partial_trace(rho, keep=[...], dims=[...])

    Parameters
    ----------
    rho_AB : ndarray (4, 4)  — estado bipartido AB
    rho_AC : ndarray (4, 4)  — estado bipartido AC
    tol : float
        Tolerância numérica para compatibilidade.

    Returns
    -------
    delta : float
        Norma de Frobenius da diferença das marginais em A.
    compatible : bool
        True se delta < tol.
    """
    # Tr_B(rho_AB): mantém subsistema 0 (A), traça 1 (B)  — dims=[2,2]
    rho_A_from_AB = partial_trace(rho_AB, keep=[0], dims=[2, 2])
    # Tr_C(rho_AC): mantém subsistema 0 (A), traça 1 (C)  — dims=[2,2]
    rho_A_from_AC = partial_trace(rho_AC, keep=[0], dims=[2, 2])

    delta = float(np.linalg.norm(rho_A_from_AB - rho_A_from_AC, 'fro'))
    compatible = delta < tol

    return delta, compatible


# ── Teste rápido de sanidade ──────────────────────────────────────────────────
print("Teste de sanidade: build_product_marginals")
rng_t = np.random.default_rng(seed=42)
rho_A_t = random_density_matrix(2, rng=rng_t)
rho_AB_t, rho_AC_t = build_product_marginals(rho_A_t, rng=rng_t)

rho_A_from_AB_t = partial_trace(rho_AB_t, keep=[0], dims=[2, 2])
rho_A_from_AC_t = partial_trace(rho_AC_t, keep=[0], dims=[2, 2])
err_AB = np.linalg.norm(rho_A_from_AB_t - rho_A_t, 'fro')
err_AC = np.linalg.norm(rho_A_from_AC_t - rho_A_t, 'fro')
print(f"  ||Tr_B(rho_AB) - rho_A||_F = {err_AB:.2e}   (esperado < 1e-14)")
print(f"  ||Tr_C(rho_AC) - rho_A||_F = {err_AC:.2e}   (esperado < 1e-14)")

print("\nTeste de sanidade: build_correlated_marginals")
rho_AB_c, rho_AC_c = build_correlated_marginals(rho_A_t, rng=rng_t)
rho_A_from_AB_c = partial_trace(rho_AB_c, keep=[0], dims=[2, 2])
rho_A_from_AC_c = partial_trace(rho_AC_c, keep=[0], dims=[2, 2])
err_AB_c = np.linalg.norm(rho_A_from_AB_c - rho_A_t, 'fro')
err_AC_c = np.linalg.norm(rho_A_from_AC_c - rho_A_t, 'fro')
print(f"  ||Tr_B(rho_AB) - rho_A||_F = {err_AB_c:.2e}")
print(f"  ||Tr_C(rho_AC) - rho_A||_F = {err_AC_c:.2e}")
delta_t, comp_t = check_overlap_consistency(rho_AB_c, rho_AC_c)
print(f"  delta_overlap = {delta_t:.2e},  overlap_compatible = {comp_t}")


## Célula 4 — Experimento A: Marginais Completamente Aleatórias

Para cada realização:
- $\rho_{AB}$ e $\rho_{AC}$ são geradas de forma completamente independente
- Verifica-se a condição de sobreposição
- O SDP de representabilidade global é resolvido independentemente

**Hipótese esperada**: a maioria dos pares aleatórios viola a condição de sobreposição e é também incompatível globalmente, mas os dois testes são independentes.

In [ ]:
# ==============================================================================
# Célula 4 — Experimento A: Marginais completamente aleatórias
# ==============================================================================

SEED_A = 2024_09_25
N_A    = 1000          # número de realizações
TOL_OVERLAP = 1e-6    # tolerância para consistência na sobreposição
DIMS_3Q = [2, 2, 2]   # 3 qubits

rng_A = np.random.default_rng(seed=SEED_A)
results_A: List[Dict[str, Any]] = []

print(f"Experimento A — Marginais aleatórias (N={N_A} realizações)")
print(f"Tolerância de sobreposição: tol = {TOL_OVERLAP:.0e}")
print("─" * 70)

t_total_A = time.perf_counter()

for i in range(N_A):
    # ── 1. Geração das marginais aleatórias ───────────────────────────────────
    rho_AB = random_density_matrix(4, rng=rng_A)
    rho_AC = random_density_matrix(4, rng=rng_A)

    # ── 2. Teste de consistência na sobreposição A ────────────────────────────
    delta_overlap, overlap_compatible = check_overlap_consistency(
        rho_AB, rho_AC, tol=TOL_OVERLAP
    )

    # ── 3. SDP de representabilidade global (independente do teste de overlap) ─
    marginals = {"AB": rho_AB, "AC": rho_AC}

    t0 = time.perf_counter()
    try:
        sdp, M, b = build_qmp_sdp(marginals, dims=DIMS_3Q)
        res = solve_qmp(sdp, solver="CLARABEL")
        solver_status = res["status"]
        is_feasible   = res["is_feasible"]
    except Exception as exc:
        solver_status = "error"
        is_feasible   = False
    solver_time = time.perf_counter() - t0

    # ── 4. Registro dos resultados ─────────────────────────────────────────────
    results_A.append({
        "experiment":         "A_random",
        "realization":        i,
        "delta_overlap":      delta_overlap,
        "overlap_compatible": overlap_compatible,
        "solver_status":      solver_status,
        "is_feasible":        is_feasible,
        "solver_time":        solver_time,
    })

    # Progresso a cada 100 amostras
    if (i + 1) % 100 == 0:
        n_overlap = sum(r["overlap_compatible"] for r in results_A)
        n_feasible = sum(r["is_feasible"] for r in results_A)
        t_el = time.perf_counter() - t_total_A
        print(
            f"  [{i+1:4d}/{N_A}]  overlap_ok={n_overlap:4d}  "
            f"feasible={n_feasible:4d}  elapsed={t_el:.1f}s"
        )

t_elapsed_A = time.perf_counter() - t_total_A
df_A = pd.DataFrame(results_A)

print("─" * 70)
print(f"Experimento A concluído em {t_elapsed_A:.2f}s")
print(f"  Total amostras:               {len(df_A)}")
print(f"  Overlap compatível:            {df_A['overlap_compatible'].sum()}")
print(f"  Globalmente factível:          {df_A['is_feasible'].sum()}")


## Célula 5 — Experimento B: Marginais com Condição de Sobreposição Satisfeita

Neste experimento, **garantimos por construção** que $\text{Tr}_B(\rho_{AB}) = \text{Tr}_C(\rho_{AC}) = \rho_A$.

Duas variantes:
- **B.prod**: construção produto $\rho_{AB} = \rho_A \otimes \sigma_B$ (admite extensão trivial)
- **B.corr**: construção correlacionada via purificação (pode ser incompatível globalmente)

**Hipótese esperada**: a variante produto é sempre factível; a correlacionada pode ser infactível, demonstrando que *overlap* ≠ *compatibilidade global*.

In [ ]:
# ==============================================================================
# Célula 5 — Experimento B: Marginais com sobreposição garantida
# ==============================================================================

SEED_B = 2024_09_26
N_B    = 1000

rng_B = np.random.default_rng(seed=SEED_B)
results_B: List[Dict[str, Any]] = []

print(f"Experimento B — Marginais com sobreposição garantida (N={N_B} por variante)")
print("─" * 70)

t_total_B = time.perf_counter()

for i in range(N_B):
    # Gera rho_A comum
    rho_A = random_density_matrix(2, rng=rng_B)

    for variant, build_fn, build_kwargs in [
        ("B.prod", build_product_marginals,    {}),
        ("B.corr", build_correlated_marginals, {}),
    ]:
        # ── 1. Constrói as marginais ──────────────────────────────────────────
        rho_AB, rho_AC = build_fn(rho_A, rng=rng_B, **build_kwargs)

        # ── 2. Verificação numérica das marginais em A ──────────────────────
        rho_A_from_AB = partial_trace(rho_AB, keep=[0], dims=[2, 2])
        rho_A_from_AC = partial_trace(rho_AC, keep=[0], dims=[2, 2])
        err_AB = float(np.linalg.norm(rho_A_from_AB - rho_A, 'fro'))
        err_AC = float(np.linalg.norm(rho_A_from_AC - rho_A, 'fro'))

        # ── 3. Teste de sobreposição ────────────────────────────────────────
        delta_overlap, overlap_compatible = check_overlap_consistency(
            rho_AB, rho_AC, tol=TOL_OVERLAP
        )

        # ── 4. SDP de representabilidade global ─────────────────────────────
        marginals = {"AB": rho_AB, "AC": rho_AC}

        t0 = time.perf_counter()
        try:
            sdp, M, b = build_qmp_sdp(marginals, dims=DIMS_3Q)
            res = solve_qmp(sdp, solver="CLARABEL")
            solver_status = res["status"]
            is_feasible   = res["is_feasible"]
        except Exception as exc:
            solver_status = "error"
            is_feasible   = False
        solver_time = time.perf_counter() - t0

        # ── 5. Registro ──────────────────────────────────────────────────────
        results_B.append({
            "experiment":         variant,
            "realization":        i,
            "delta_overlap":      delta_overlap,
            "overlap_compatible": overlap_compatible,
            "marginal_err_AB":    err_AB,
            "marginal_err_AC":    err_AC,
            "solver_status":      solver_status,
            "is_feasible":        is_feasible,
            "solver_time":        solver_time,
        })

    # Progresso
    if (i + 1) % 200 == 0:
        n_prod_feas = sum(
            r["is_feasible"] for r in results_B if r["experiment"] == "B.prod"
        )
        n_corr_feas = sum(
            r["is_feasible"] for r in results_B if r["experiment"] == "B.corr"
        )
        t_el = time.perf_counter() - t_total_B
        print(
            f"  [{i+1:4d}/{N_B}]  prod_feasible={n_prod_feas:4d}  "
            f"corr_feasible={n_corr_feas:4d}  elapsed={t_el:.1f}s"
        )

t_elapsed_B = time.perf_counter() - t_total_B
df_B = pd.DataFrame(results_B)

print("─" * 70)
print(f"Experimento B concluído em {t_elapsed_B:.2f}s")
for variant in ["B.prod", "B.corr"]:
    sub = df_B[df_B["experiment"] == variant]
    print(f"  Variante {variant}:")
    print(f"    Total amostras:     {len(sub)}")
    print(f"    Overlap compatível: {sub['overlap_compatible'].sum()} ({100*sub['overlap_compatible'].mean():.1f}%)")
    print(f"    Globalmente fact.:  {sub['is_feasible'].sum()} ({100*sub['is_feasible'].mean():.1f}%)")
    print(f"    Err AB médio:       {sub['marginal_err_AB'].mean():.2e}")
    print(f"    Err AC médio:       {sub['marginal_err_AC'].mean():.2e}")


## Célula 6 — Caso Especial: Incompatibilidade de Bell (Validação)

O estado de Bell $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ satisfaz:
$$\text{Tr}_B(|\Phi^+\rangle\langle\Phi^+|) = \frac{I}{2} = \text{Tr}_C(|\Phi^+\rangle\langle\Phi^+|)$$

Logo a **condição de sobreposição é satisfeita**, mas pelo **monogamia do entrelaçamento**,
não existe $\rho_{ABC}$ tal que $\text{Tr}_C(\rho_{ABC}) = |\Phi^+\rangle\langle\Phi^+|_{AB}$
e $\text{Tr}_B(\rho_{ABC}) = |\Phi^+\rangle\langle\Phi^+|_{AC}$ simultaneamente.

Este é o **caso de validação fundamental** do experimento.

In [ ]:
# ==============================================================================
# Célula 6 — Caso especial: Incompatibilidade de Bell (caso de validação)
# ==============================================================================

print("=" * 65)
print("CASO DE VALIDAÇÃO: Incompatibilidade de Marginais de Bell")
print("=" * 65)

# ── 1. Construção do estado de Bell |Phi+> ────────────────────────────────────
phi_plus = (np.kron(zero, zero) + np.kron(one, one)) / np.sqrt(2.0)
rho_bell = ketbra(phi_plus)   # reutiliza ketbra do projeto

print("\n|Phi+> = (|00> + |11>) / sqrt(2)")
print(f"Norma de |Phi+>: {np.linalg.norm(phi_plus):.6f} (deve ser 1.0)")

# ── 2. Prescrição das marginais ───────────────────────────────────────────────
marginals_bell = {
    "AB": rho_bell,
    "AC": rho_bell,
}

# ── 3. Verificação das marginais em A ─────────────────────────────────────────
rho_A_from_bell_AB = partial_trace(rho_bell, keep=[0], dims=[2, 2])
rho_A_from_bell_AC = partial_trace(rho_bell, keep=[0], dims=[2, 2])

print("\nMarginal Tr_B(rho_bell):")
print(np.round(rho_A_from_bell_AB, 6))
print("\nMarginal Tr_C(rho_bell):")
print(np.round(rho_A_from_bell_AC, 6))

I_over_2 = np.eye(2, dtype=complex) / 2.0
err_Id_AB = float(np.linalg.norm(rho_A_from_bell_AB - I_over_2, 'fro'))
err_Id_AC = float(np.linalg.norm(rho_A_from_bell_AC - I_over_2, 'fro'))
print(f"\n||Tr_B(rho_bell) - I/2||_F = {err_Id_AB:.2e}  (esperado ≈ 0)")
print(f"||Tr_C(rho_bell) - I/2||_F = {err_Id_AC:.2e}  (esperado ≈ 0)")

# ── 4. Teste de sobreposição ──────────────────────────────────────────────────
delta_bell, overlap_bell = check_overlap_consistency(
    rho_bell, rho_bell, tol=TOL_OVERLAP
)
print(f"\ndelta_overlap = {delta_bell:.2e}")
print(f"overlap_compatible = {overlap_bell}  ✓ (condição necessária satisfeita)")

# ── 5. SDP de representabilidade global ──────────────────────────────────────
print("\nResolvendo SDP de representabilidade global...")
t0_bell = time.perf_counter()
sdp_bell, M_bell, b_bell = build_qmp_sdp(marginals_bell, dims=DIMS_3Q)
res_bell = solve_qmp(sdp_bell, solver="CLARABEL")
t_bell = time.perf_counter() - t0_bell

print(f"Status do solver:       {res_bell['status']}")
print(f"Problema factível:      {res_bell['is_feasible']}")
print(f"Mensagem de erro:       {res_bell['error_message']}")
print(f"Tempo de resolução:     {t_bell:.4f}s")

# ── 6. Interpretação física ───────────────────────────────────────────────────
print("\n" + "─" * 65)
if not res_bell["is_feasible"]:
    print("✓ VALIDAÇÃO CONFIRMADA:")
    print("  A condição de sobreposição é SATISFEITA (Tr_B = Tr_C = I/2),")
    print("  mas NÃO EXISTE extensão global rho_ABC.")
    print("  Isso demonstra: overlap compatibility ≠ global representability.")
    print("  Causa física: monogamia do entrelaçamento quântico.")
else:
    print("✗ ATENÇÃO: O solver retornou factível — verifique a implementação.")
print("─" * 65)

# ── 7. Adiciona ao DataFrame para análise consolidada ─────────────────────────
result_bell = {
    "experiment":         "Bell_validation",
    "realization":        0,
    "delta_overlap":      delta_bell,
    "overlap_compatible": overlap_bell,
    "solver_status":      res_bell["status"],
    "is_feasible":        res_bell["is_feasible"],
    "solver_time":        t_bell,
}
df_bell = pd.DataFrame([result_bell])
print("\nResumo do caso Bell:")
print(df_bell.to_string(index=False))


## Célula 7 — Estatísticas Consolidadas

Análise estatística completa de todos os experimentos.

In [ ]:
# ==============================================================================
# Célula 7 — Estatísticas consolidadas
# ==============================================================================

# Combina todos os resultados em um único DataFrame
df_all = pd.concat([df_A, df_B, df_bell], ignore_index=True)

# Garante tipos corretos
df_all["overlap_compatible"] = df_all["overlap_compatible"].astype(bool)
df_all["is_feasible"]        = df_all["is_feasible"].astype(bool)


def print_experiment_stats(df: pd.DataFrame, label: str) -> Dict[str, Any]:
    """
    Imprime estatísticas de um subconjunto do DataFrame e retorna um dicionário
    com as métricas calculadas.
    """
    n_total    = len(df)
    n_overlap  = int(df["overlap_compatible"].sum())
    n_feasible = int(df["is_feasible"].sum())

    # Pares que satisfazem overlap mas são globalmente incompatíveis
    mask_overlap_infeasible = df["overlap_compatible"] & ~df["is_feasible"]
    n_overlap_infeasible    = int(mask_overlap_infeasible.sum())

    # Pares que satisfazem overlap e são globalmente compatíveis
    mask_overlap_feasible   = df["overlap_compatible"] & df["is_feasible"]
    n_overlap_feasible      = int(mask_overlap_feasible.sum())

    # P(globalmente compatível | overlap compatível)
    p_cond = n_overlap_feasible / n_overlap if n_overlap > 0 else float("nan")

    print(f"\n{'═'*65}")
    print(f"  Experimento: {label}")
    print(f"{'═'*65}")
    print(f"  Total de amostras:                        {n_total:6d}")
    print(f"  Overlap compatível:                       {n_overlap:6d}  ({100*n_overlap/n_total:.1f}%)")
    print(f"  Globalmente factível:                     {n_feasible:6d}  ({100*n_feasible/n_total:.1f}%)")
    print(f"  Overlap ✓  e  SDP infactível:             {n_overlap_infeasible:6d}")
    print(f"  Overlap ✓  e  SDP factível:               {n_overlap_feasible:6d}")
    if n_overlap > 0:
        print(f"  P(factível | overlap ✓):                  {p_cond:.4f}")
    else:
        print(f"  P(factível | overlap ✓):                  N/A (sem amostras com overlap)")
    print(f"  delta_overlap — média:                    {df['delta_overlap'].mean():.4e}")
    print(f"  delta_overlap — mediana:                  {df['delta_overlap'].median():.4e}")
    print(f"  solver_time   — média (s):                {df['solver_time'].mean():.4f}")

    return {
        "label":                label,
        "n_total":              n_total,
        "n_overlap":            n_overlap,
        "n_feasible":           n_feasible,
        "n_overlap_infeasible": n_overlap_infeasible,
        "n_overlap_feasible":   n_overlap_feasible,
        "p_cond_feasible_given_overlap": p_cond,
    }


print("RESUMO ESTATÍSTICO COMPLETO")
stats_summary = []

for exp_name in df_all["experiment"].unique():
    sub = df_all[df_all["experiment"] == exp_name]
    s = print_experiment_stats(sub, exp_name)
    stats_summary.append(s)

# Tabela resumo
print("\n" + "=" * 90)
print("TABELA RESUMO")
print("=" * 90)
df_stats = pd.DataFrame(stats_summary)
print(df_stats.to_string(index=False))

# ── Análise focada: sobreposição ≠ compatibilidade global ─────────────────────
print("\n" + "=" * 65)
print("CONCLUSÃO PRINCIPAL: overlap ≠ representabilidade global")
print("=" * 65)

for row in stats_summary:
    if row["n_overlap_infeasible"] > 0:
        pct = 100 * row["n_overlap_infeasible"] / row["n_overlap"] if row["n_overlap"] > 0 else 0
        print(
            f"  [{row['label']}] {row['n_overlap_infeasible']} par(es) com "
            f"overlap ✓ mas SDP infactível ({pct:.1f}% dos com overlap)"
        )


## Célula 8 — Visualizações

Quatro gráficos principais:
1. Histograma de `delta_overlap` para marginais aleatórias
2. Comparação de `overlap_compatible` vs `is_feasible` por experimento
3. Distribuição de `delta_overlap` separando casos compatíveis e incompatíveis
4. Contagem dos 4 quadrantes lógicos (overlap × SDP)

In [ ]:
# ==============================================================================
# Célula 8 — Visualizações
# ==============================================================================

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {
    "overlap_true":  "#2ecc71",   # verde
    "overlap_false": "#e74c3c",   # vermelho
    "feasible":      "#3498db",   # azul
    "infeasible":    "#e67e22",   # laranja
    "neutral":       "#95a5a6",   # cinza
}

fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    "Compatibilidade de Marginais Quânticas para 3 Qubits\n"
    r"Overlap Consistency $\neq$ Global Representability",
    fontsize=15, fontweight="bold", y=0.98
)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 1: Histograma de delta_overlap — Experimento A (marginais aleatórias)
# ─────────────────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)

data_A_delta = df_A["delta_overlap"].values
ax1.hist(
    data_A_delta, bins=50,
    color=COLORS["neutral"], edgecolor="white", linewidth=0.5
)
ax1.axvline(
    TOL_OVERLAP, color="red", linestyle="--", linewidth=1.5,
    label=f"tol = {TOL_OVERLAP:.0e}"
)
ax1.set_xlabel(r"$\|\mathrm{Tr}_B(\rho_{AB}) - \mathrm{Tr}_C(\rho_{AC})\|_F$", fontsize=11)
ax1.set_ylabel("Frequência", fontsize=11)
ax1.set_title("Exp. A: Histograma de delta_overlap\n(marginais aleatórias)", fontsize=11)
ax1.legend(fontsize=10)

n_left = (data_A_delta < TOL_OVERLAP).sum()
ax1.text(
    0.02, 0.97,
    f"{n_left} pares com\noverlap < tol",
    transform=ax1.transAxes, va="top", fontsize=9,
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.7)
)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 2: Barras — overlap_compatible vs is_feasible por experimento
# ─────────────────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)

exp_labels = []
pct_overlap = []
pct_feasible = []

for exp_name in ["A_random", "B.prod", "B.corr"]:
    sub = df_all[df_all["experiment"] == exp_name]
    exp_labels.append(exp_name)
    pct_overlap.append(100 * sub["overlap_compatible"].mean())
    pct_feasible.append(100 * sub["is_feasible"].mean())

x_pos = np.arange(len(exp_labels))
width = 0.35

bars1 = ax2.bar(
    x_pos - width/2, pct_overlap, width,
    label="Overlap compatível", color=COLORS["overlap_true"], alpha=0.85
)
bars2 = ax2.bar(
    x_pos + width/2, pct_feasible, width,
    label="SDP factível", color=COLORS["feasible"], alpha=0.85
)

for bar in bars1:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 1, f"{h:.1f}%",
             ha="center", va="bottom", fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 1, f"{h:.1f}%",
             ha="center", va="bottom", fontsize=8)

ax2.set_xticks(x_pos)
ax2.set_xticklabels(exp_labels, fontsize=10)
ax2.set_ylabel("% de amostras", fontsize=11)
ax2.set_title("Overlap vs Factibilidade\npor experimento", fontsize=11)
ax2.set_ylim(0, 115)
ax2.legend(fontsize=9)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 3: Distribuição de delta_overlap — casos compatíveis vs incompatíveis
#            (Exp. B.corr — mais interessante)
# ─────────────────────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)

df_Bcorr = df_all[df_all["experiment"] == "B.corr"]
delta_Bcorr_feasible   = df_Bcorr[df_Bcorr["is_feasible"] == True]["delta_overlap"].values
delta_Bcorr_infeasible = df_Bcorr[df_Bcorr["is_feasible"] == False]["delta_overlap"].values

bins_bc = np.linspace(0, max(df_Bcorr["delta_overlap"].max(), 1e-10), 40)

if len(delta_Bcorr_feasible) > 0:
    ax3.hist(
        delta_Bcorr_feasible, bins=bins_bc, alpha=0.6,
        label=f"SDP factível (n={len(delta_Bcorr_feasible)})",
        color=COLORS["feasible"]
    )
if len(delta_Bcorr_infeasible) > 0:
    ax3.hist(
        delta_Bcorr_infeasible, bins=bins_bc, alpha=0.6,
        label=f"SDP infactível (n={len(delta_Bcorr_infeasible)})",
        color=COLORS["infeasible"]
    )

ax3.axvline(TOL_OVERLAP, color="red", linestyle="--", linewidth=1.5, label="tol")
ax3.set_xlabel(r"$\delta_{\mathrm{overlap}}$", fontsize=11)
ax3.set_ylabel("Frequência", fontsize=11)
ax3.set_title(
    "Exp. B.corr: delta_overlap\nfactível vs infactível", fontsize=11
)
ax3.legend(fontsize=9)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 4: Quadrantes lógicos — overlap × SDP
# ─────────────────────────────────────────────────────────────────────────────
ax4 = fig.add_subplot(2, 3, 4)

quadrant_data = {}
for exp_name in ["A_random", "B.prod", "B.corr"]:
    sub = df_all[df_all["experiment"] == exp_name]
    quadrant_data[exp_name] = {
        "OV=T, SDP=T":  int((sub["overlap_compatible"] &  sub["is_feasible"]).sum()),
        "OV=T, SDP=F":  int((sub["overlap_compatible"] & ~sub["is_feasible"]).sum()),
        "OV=F, SDP=T":  int((~sub["overlap_compatible"] &  sub["is_feasible"]).sum()),
        "OV=F, SDP=F":  int((~sub["overlap_compatible"] & ~sub["is_feasible"]).sum()),
    }

quad_labels = ["OV=T\nSDP=T", "OV=T\nSDP=F", "OV=F\nSDP=T", "OV=F\nSDP=F"]
quad_colors = [
    COLORS["feasible"],
    COLORS["infeasible"],
    COLORS["neutral"],
    COLORS["overlap_false"],
]

exp_names_q = ["A_random", "B.prod", "B.corr"]
x_q = np.arange(len(quad_labels))
bar_width = 0.25

for k, exp_name in enumerate(exp_names_q):
    vals = [quadrant_data[exp_name][q.replace("\n", "=".join(["OV", "SDP"]))
                                     .replace("\n", ", ")]
            for q in ["OV=T, SDP=T", "OV=T, SDP=F", "OV=F, SDP=T", "OV=F, SDP=F"]]
    ax4.bar(
        x_q + k * bar_width - bar_width, vals, bar_width,
        label=exp_name, alpha=0.8,
        color=["#2ecc71", "#e67e22", "#3498db", "#e74c3c"][0],  # placeholder color
    )

# Redesenha com cores corretas por quadrante
ax4.cla()
offsets = [-bar_width, 0, bar_width]
exp_plot_colors = ["#2ecc71", "#3498db", "#e67e22"]

for k, (exp_name, color) in enumerate(zip(exp_names_q, exp_plot_colors)):
    vals = [
        quadrant_data[exp_name]["OV=T, SDP=T"],
        quadrant_data[exp_name]["OV=T, SDP=F"],
        quadrant_data[exp_name]["OV=F, SDP=T"],
        quadrant_data[exp_name]["OV=F, SDP=F"],
    ]
    bars_q = ax4.bar(
        x_q + offsets[k], vals, bar_width,
        label=exp_name, color=color, alpha=0.82, edgecolor="white"
    )
    for bar, v in zip(bars_q, vals):
        if v > 0:
            ax4.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
                str(v), ha="center", va="bottom", fontsize=7
            )

ax4.set_xticks(x_q)
ax4.set_xticklabels(quad_labels, fontsize=10)
ax4.set_ylabel("Número de amostras", fontsize=11)
ax4.set_title(
    "Quadrantes: Overlap × Factibilidade SDP",
    fontsize=11
)
ax4.legend(fontsize=9, loc="upper right")

# Adiciona anotação de destaque no quadrante crítico OV=T, SDP=F
ax4.annotate(
    "← Região crítica:\noverlap ✓ mas\nextensão global ✗",
    xy=(1, max(quadrant_data[exp]["OV=T, SDP=F"] for exp in exp_names_q) + 5),
    xytext=(2, max(quadrant_data[exp]["OV=T, SDP=F"] for exp in exp_names_q) + 50),
    arrowprops=dict(arrowstyle="->", color="black"),
    fontsize=9, color="darkred"
)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 5: Histograma de delta_overlap — Experimentos B.prod e B.corr
# ─────────────────────────────────────────────────────────────────────────────
ax5 = fig.add_subplot(2, 3, 5)

for exp_name, color, ls in [
    ("B.prod", "#3498db", "-"),
    ("B.corr", "#e67e22", "--"),
]:
    sub = df_all[df_all["experiment"] == exp_name]
    ax5.hist(
        sub["delta_overlap"].values,
        bins=50, alpha=0.6, label=exp_name,
        color=color, edgecolor="white", linewidth=0.5
    )

ax5.axvline(TOL_OVERLAP, color="red", linestyle="--", linewidth=1.5, label=f"tol={TOL_OVERLAP:.0e}")
ax5.set_xlabel(r"$\delta_{\mathrm{overlap}}$", fontsize=11)
ax5.set_ylabel("Frequência", fontsize=11)
ax5.set_title("Exp. B: delta_overlap\n(produto vs correlacionado)", fontsize=11)
ax5.legend(fontsize=9)
ax5.set_xscale("symlog", linthresh=1e-10)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 6: Tempo de resolução do SDP por experimento (boxplot)
# ─────────────────────────────────────────────────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)

exp_for_time = ["A_random", "B.prod", "B.corr"]
time_data = [
    df_all[df_all["experiment"] == exp]["solver_time"].values
    for exp in exp_for_time
]

bp = ax6.boxplot(
    time_data, labels=exp_for_time,
    patch_artist=True, notch=False,
    medianprops=dict(color="black", linewidth=2)
)
box_colors = ["#2ecc71", "#3498db", "#e67e22"]
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax6.set_ylabel("Tempo de resolução (s)", fontsize=11)
ax6.set_title("Tempo do Solver CLARABEL\npor experimento", fontsize=11)
ax6.set_yscale("log")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("experimento_compatibilidade_marginais.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✓ Figura salva: experimento_compatibilidade_marginais.png")


In [ ]:
# ==============================================================================
# Célula 9 — Conclusão e resumo dos arquivos
# ==============================================================================

print("=" * 70)
print("RESUMO DO EXPERIMENTO")
print("=" * 70)
print("""
Este notebook implementou um estudo numérico da região de compatibilidade
de marginais quânticas para sistemas de 3 qubits.

FUNÇÕES REUTILIZADAS (de inicial_teste.ipynb, sem modificações):
  - ketbra(psi)                   → projetor |psi><psi|
  - tensor(*args)                 → produto de Kronecker sequencial
  - partial_trace(rho, keep, dims)→ traço parcial via einsum
  - base_hermitiana(d)            → base de Herm(d)
  - embed_operator(O, sys, dim)   → embedding de operador local
  - parse_subsystem_key(key)      → normalização de chaves de subsistema
  - build_qmp_sdp(marginals, dims)→ montagem do SDP (SDPLab)
  - solve_qmp(sdp, solver)        → solução do SDP via CLARABEL

NOVAS FUNÇÕES (específicas deste experimento):
  - random_density_matrix(d, rng)         → estado aleatório (Ginibre)
  - build_product_marginals(rho_A, rng)   → rho_AB=rho_A⊗sigma_B (trivial)
  - build_correlated_marginals(rho_A,rng) → purificação + canal aleatório
  - check_overlap_consistency(rho_AB, rho_AC, tol)
  - print_experiment_stats(df, label)     → estatísticas consolidadas

EXPERIMENTOS:
  - Exp. A  (N=1000): marginais completamente aleatórias
  - Exp. B.prod (N=1000): rho_AB = rho_A ⊗ sigma_B (sobreposição garantida)
  - Exp. B.corr (N=1000): purificação correlacionada (sobreposição garantida)
  - Bell (N=1): caso de validação — overlap ✓ mas SDP infactível

RESULTADO PRINCIPAL:
  A consistência na sobreposição (Tr_B(rho_AB) = Tr_C(rho_AC)) é condição
  NECESSÁRIA mas NÃO SUFICIENTE para existência de extensão global rho_ABC.
  O caso de Bell e os casos B.corr demonstram isso numericamente.
""")

# Exibe o DataFrame final com todos os resultados
print("Primeiras 5 linhas do DataFrame completo:")
print(df_all[["experiment", "delta_overlap", "overlap_compatible",
              "solver_status", "is_feasible", "solver_time"]].head())
